# RiskModels — from HTTP to “why did it move?”

You are about to climb three rungs of the same ladder:

| Rung | You type… | Best for… |
|------|-----------|-----------|
| **1. REST** | `requests` + URLs | Debugging, any language, seeing the raw contract |
| **2. SDK** | `RiskModelsClient` methods | Typed helpers, pandas, validation, `df.attrs["legend"]` |
| **3. AOM** | `rm().subject(...).scope(...).return_attribution(...)` | Analyst questions, composable plans, agents |

**Three numbers professionals actually argue about** (we touch all three in this notebook):

- **Return** — how much came from market / sector / subsector sleeves vs the name (cumulative factor attribution).
- **Risk** — share of variance explained at each level (**ER** is a fraction from 0 to 1; at L3 the explained terms + residual ≈ 1).
- **Hedge ratio (HR)** — **dollars of ETF per one dollar of stock** (HR can be negative when orthogonality points the other way).

If that sounds dry — good: the API is the antidote. Paste a key, run a cell, and you are staring at a live decomposition instead of a static deck.

**[Get your API key →](https://riskmodels.app/get-key)** · **[ERM3 methodology (wiki)](https://riskmodels.net/docs/methodology/erm3-l3)** · **[Open in Colab](https://colab.research.google.com/github/BlueWaterCorp/RiskModels_API/blob/main/sdk/notebooks/riskmodels_quickstart.ipynb)** · Notebook path: `sdk/notebooks/riskmodels_quickstart.ipynb`.

**Map:** optional Colab installs → **Part 1 (REST: returns chart + early L3 Plotly)** → **Part 2 (SDK + AOM, including a chain)** → **Part 3 (REST cookbook)**. Keys and base URL use `riskmodels.notebook.quickstart_connect()`.


### Colab only (skip on local Jupyter)

Run the next cell once. It installs **`riskmodels-py[viz]`** (Plotly + notebook helpers) from **PyPI**, or **GitHub `main`** if that version is not on PyPI yet—so the same notebook works for clients without a manual fix.


In [ ]:
import sys

try:
    import google.colab  # noqa: F401

    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    import subprocess

    _deps = ["requests", "python-dotenv"]
    _pypi = "riskmodels-py[viz]>=0.3.3,<0.4"
    _git = (
        "riskmodels-py[viz] @ git+https://github.com/BlueWaterCorp/RiskModels_API.git"
        "@main#subdirectory=sdk"
    )
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _pypi, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print("Colab: installed riskmodels-py[viz] from PyPI.")
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", _git, *_deps],
            stdout=subprocess.DEVNULL,
        )
        print(
            "Colab: installed riskmodels-py[viz] from GitHub main "
            "(PyPI may not list this version yet)."
        )
else:
    print("Local: using your environment (install sdk + requests + python-dotenv if needed).")


---
## Part 1 — Raw REST (see the wire format)

**Why start here?** Because once you have seen the JSON keys and query parameters, nothing the SDK or AOM does feels like magic — it is all the same endpoints, wrapped.

In this part you will call **`GET /ticker-returns`** and build the **return** attribution picture (cumulative sleeves). **Risk (ER)** and **hedge ratios (HR)** show up in **`GET /metrics/{ticker}`** — we reuse those calls again in Part 3 so you can compare raw JSON vs the SDK’s semantic column names.

`quickstart_connect()` loads your key (`.env.local`, shell `export`, Colab Secret, or prompt) and returns a authenticated `requests` session plus `BASE_URL` ending in `/api`.

After the first chart you will load an **L3 variance** Plotly figure (still pure REST + one SDK visual helper) so the “risk” story appears early, not only as a table at the end.


In [ ]:
# ── 1a. Connect — keys & session via riskmodels.notebook (then pure REST below) ─────
import os

import pandas as pd
from IPython.display import display, Image

from riskmodels.notebook import quickstart_connect

session, BASE_URL, API_KEY = quickstart_connect()


### Precision hedge chart (REST) — **Return** attribution

`GET /ticker-returns` returns daily **gross** returns plus **combined factor return** (CFR) sleeves at L1/L2/L3. Cumulating those series answers: *how much of the stock path lived at market, sector, and subsector?* (**Risk** and scalar **HR/ER** for the *latest* date show up in Part 3 / `get_metrics`.)


In [ ]:
# ── Precision Hedge Chart ────────────────────────────────────────────────────────────
# Embed figures in the notebook (Part 2 also calls `%matplotlib inline`; this cell runs first in order).
from IPython import get_ipython

_ipy = get_ipython()
if _ipy is not None:
    _ipy.run_line_magic("matplotlib", "inline")

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

TICKER = "NVDA"   # Try it: change to "AAPL", "TSLA", or "JPM"
YEARS  = 5        # Try it: set to 1 or 5 to zoom in/out

resp = session.get(
    f"{BASE_URL}/ticker-returns",
    params={"ticker": TICKER, "years": YEARS},
)
resp.raise_for_status()
data = resp.json().get("data", [])
df = pd.DataFrame(data)

if df.empty:
    print(f"No data for {TICKER}.")
else:
    df["date"] = pd.to_datetime(df["date"])
    df = df.sort_values("date").reset_index(drop=True)

    def cumulative_decimal(r: pd.Series) -> np.ndarray:
        x = np.asarray(r.fillna(0.0), dtype=float)
        return np.cumprod(1.0 + x) - 1.0

    df["cum_stock"]     = cumulative_decimal(df["returns_gross"])
    df["cum_market"]    = cumulative_decimal(df["l1_cfr"])
    df["cum_sector"]    = cumulative_decimal(df["l2_cfr"])
    df["cum_subsector"] = cumulative_decimal(df["l3_cfr"])

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor("#0d0d0d")
    ax.set_facecolor("#0d0d0d")

    colors = {
        "cum_stock":     "#60a5fa",
        "cum_market":    "#6366f1",
        "cum_sector":    "#34d399",
        "cum_subsector": "#9ca3af",
    }
    labels = {
        "cum_stock":     TICKER,
        "cum_market":    "Market (L1)",
        "cum_sector":    "Sector (L2)",
        "cum_subsector": "Subsector (L3)",
    }

    for col, color in colors.items():
        ax.plot(df["date"], df[col], color=color, linewidth=1.4, label=labels[col])

    ax.axhline(0, color="#444", linewidth=0.8, linestyle="--")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0, decimals=0))
    ax.tick_params(colors="#aaa", labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor("#333")
    ax.set_xlabel("Date", color="#aaa", fontsize=9)
    ax.set_ylabel("Cumulative Return", color="#aaa", fontsize=9)
    ax.set_title(
        f"Your Precision Hedge Recipe — {TICKER}  ({YEARS}y)",
        color="white", fontsize=12, pad=10
    )
    ax.legend(
        frameon=False, labelcolor="#ccc", fontsize=9,
        loc="upper left", title="Series", title_fontsize=8,
    )
    ax.grid(axis="y", color="#222", linewidth=0.6)
    plt.tight_layout()
    plt.show()

    latest = df.iloc[-1]
    summary = pd.DataFrame({
        "Value": {
            f"{TICKER} total return":    f"{latest.cum_stock * 100:.1f}%",
            "Market (L1 CFR)":           f"{latest.cum_market * 100:.1f}%",
            "Sector (L2 CFR)":           f"{latest.cum_sector * 100:.1f}%",
            "Subsector (L3 CFR)":        f"{latest.cum_subsector * 100:.1f}%",
            "Residual (approx.)": f"{(latest.cum_stock - latest.cum_subsector) * 100:.1f}%",
        }
    })
    print(f"Cumulative returns over {YEARS}y — as of {latest.date.date()}")
    display(summary)

### L3 year-end bars (Plotly) — **annual σ × ER**

Right after the **return** sleeves, this cell pulls **5y** ``GET /l3-decomposition`` and aligns **``vol_23d``** from ``GET /ticker-returns``. Each calendar year gets one **stacked bar**: segment height = **annualized σ × L3 ER** (bar height ≈ total σ, not forced to 100%). RiskModels **``vol_23d``** is already annual (daily √252-style); only set **``MONTHLY_VOL_TO_ANNUAL = True``** if you substitute a **monthly** stdev series (then σ is multiplied by **√12**). Same tabular recipe as Part 3 ``/l3-decomposition``.

If import fails, run: `pip install plotly` (or `pip install "riskmodels-py[viz]"`).

The next cell defaults to **PLTR** and joins **legend labels** to **API fields** in a small table. Human-readable **industry** text comes from optional `ticker_metadata` (set `NEXT_PUBLIC_SUPABASE_URL` + `NEXT_PUBLIC_SUPABASE_ANON_KEY` in `.env.local` — public read). ETF tickers always come from **`GET /metrics`**.


In [ ]:
# ── L3 year-end stacked bars (σ × ER), 5y — REST + vol from /ticker-returns ────────
# %pip install -q "riskmodels-py[viz]"

import os

import pandas as pd
from riskmodels import plot_l3_year_end_stack
from riskmodels.env import load_repo_dotenv

L3_HISTORY_YEARS = 5
sdk_ticker = "PLTR"  # Try it: any US equity ticker

# ``monthly_vol_to_annual=True`` multiplies σ by √12 — use only if ``vol_23d`` is *monthly* stdev.
# RiskModels ``vol_23d`` is already **annual** (≈ √252 from daily); leave False.
MONTHLY_VOL_TO_ANNUAL = False

_mresp = session.get(f"{BASE_URL}/metrics/{sdk_ticker}")
_mresp.raise_for_status()
_mbody = _mresp.json()
_meta_m = _mbody.get("meta") or {}
_sector_etf = _meta_m.get("sector_etf")
_subsector_etf = _meta_m.get("subsector_etf")

_factset_industry = None
_gics_sector = None
try:
    load_repo_dotenv()
    _sb = (os.environ.get("SUPABASE_URL") or os.environ.get("NEXT_PUBLIC_SUPABASE_URL") or "").rstrip("/")
    _key = (
        os.environ.get("NEXT_PUBLIC_SUPABASE_ANON_KEY")
        or os.environ.get("SUPABASE_ANON_KEY")
        or ""
    ).strip()
    if _sb and _key:
        _trm = session.get(
            f"{_sb}/rest/v1/ticker_metadata",
            params={
                "ticker": f"eq.{sdk_ticker.upper()}",
                "select": "factset_industry_name,gics_sector_name,subsector_etf,sector_etf",
            },
            headers={"apikey": _key, "Authorization": f"Bearer {_key}"},
            timeout=15,
        )
        if _trm.ok:
            _rows = _trm.json()
            if isinstance(_rows, list) and _rows:
                _factset_industry = (_rows[0] or {}).get("factset_industry_name")
                _gics_sector = (_rows[0] or {}).get("gics_sector_name")
except Exception:
    pass

if _factset_industry and _subsector_etf:
    _subsector_legend = f"{_factset_industry} ({_subsector_etf})"
elif _factset_industry:
    _subsector_legend = _factset_industry
elif _subsector_etf:
    _subsector_legend = f"ETF {_subsector_etf}"
else:
    _subsector_legend = "subsector assignment"

_layer_disp = {
    "market": "Market (L1 · SPY)",
    "sector": f"Sector (L2 · {_sector_etf})" if _sector_etf else "Sector (L2)",
    "subsector": f"Subsector (L3 · {_subsector_legend})",
    "residual": "Residual (idiosyncratic)",
}

_l3r = session.get(
    f"{BASE_URL}/l3-decomposition",
    params={"ticker": sdk_ticker, "market_factor_etf": "SPY", "years": L3_HISTORY_YEARS},
)
_l3r.raise_for_status()
raw_l3 = _l3r.json()

_trr = session.get(
    f"{BASE_URL}/ticker-returns",
    params={"ticker": sdk_ticker, "years": L3_HISTORY_YEARS},
)
_trr.raise_for_status()
_tr_body = _trr.json()
_tr_rows = _tr_body.get("data") or []
_tr_df = pd.DataFrame(_tr_rows)
if _tr_df.empty or "date" not in _tr_df.columns:
    _vol_aligned = None
else:
    _tr_df["_d"] = pd.to_datetime(_tr_df["date"], errors="coerce").dt.normalize()
    _l3_dates = pd.to_datetime(raw_l3.get("dates") or [], errors="coerce")
    _ldf = pd.DataFrame({"_d": _l3_dates.dt.normalize()})
    _merged = _ldf.merge(
        _tr_df[["_d", "vol_23d"]],
        on="_d",
        how="left",
    )
    _vol_aligned = _merged["vol_23d"].tolist()

fig = plot_l3_year_end_stack(
    raw_l3,
    title=f"{sdk_ticker} — L3 risk at year-end (annual σ × ER)",
    layer_display_names=_layer_disp,
    max_calendar_years=L3_HISTORY_YEARS,
    vol_23d=_vol_aligned,
    monthly_vol_to_annual=MONTHLY_VOL_TO_ANNUAL,
)
fig.show()

if fig.layout.meta.get("interpretation"):
    print(fig.layout.meta["interpretation"])

_map = dict(fig.layout.meta.get("l3_mapping") or {})
_disp = fig.layout.meta.get("layer_display_names") or {}
_footer = []
for _k, _api in _map.items():
    _footer.append({
        "Layer": _k,
        "API field": _api,
        "ERM3 sleeve (legend)": _disp.get(_k, "—"),
    })
print("Legend ↔ API field ↔ sleeve (σ from vol_23d; ×√12 only if MONTHLY_VOL_TO_ANNUAL):")
display(pd.DataFrame(_footer))


**How to read the bands (why subsector can look almost invisible)**

Each day’s stacked areas are **orthogonal shares of return variance** — they sum to **~100%** (market + sector + subsector + residual). The **grey** layer is **total** market R² at L1; **blue** is the *incremental* sector sleeve beyond that; **purple** is the *incremental* subsector sleeve beyond sector. Names that hug their sector ETF often have **very little** left for subsector, so the purple sliver is real data, not a bug. Use hover for precise %.

**Checkpoint — You now have both pictures**

- **Return:** cumulative L1/L2/L3 sleeves from `/ticker-returns`.
- **Risk:** stacked **explained variance** layers from `/l3-decomposition`.

Next: **Part 2** adds the SDK (semantic names + `df.attrs`) and **AOM** (questions, plans, and a **chain** that stitches analyze → hedge).


---
## Part 2 — Python SDK, then AOM

### 2a — SDK (methods, not URLs)

`RiskModelsClient` wraps the same REST routes. You still think in tickers and DataFrames — the client attaches ERM3 **legends** and normalizes names where it can.

### 2b — AOM (`rm` / `run`)

The **Analysis Object Model** is a *question language*: subject + scope + lens (`return_attribution`, `exposure`, …). The compiler turns that into REST steps. For more patterns see [`riskmodels_aom_colab.ipynb`](./riskmodels_aom_colab.ipynb).


In [ ]:
# ── SDK client (AOM entrypoints: rm, run, stock) ─────────────────────────────────────
from riskmodels import RiskModelsClient, rm, run
from riskmodels.aom import stock
from riskmodels.env import load_repo_dotenv

import os

load_repo_dotenv()
os.environ.setdefault("RISKMODELS_BASE_URL", BASE_URL)

client = RiskModelsClient.from_env()
print("SDK ready: client, rm, run, stock.")


**Try it — `get_metrics` (snapshot of risk + hedge ratios)**

The wire response nests scalars under `metrics`. The SDK flattens to one row and attaches **`df.attrs`** (legend, lineage, kind, cheatsheet) for LLM / agent context — the next cell **prints `attrs`** explicitly.


In [ ]:
# ── 2a. SDK — same snapshot as GET /metrics/{ticker} ────────────────────────────────
from pprint import pprint

ticker_sdk = "NVDA"  # Try it: your symbol

m_df = client.get_metrics(ticker_sdk, as_dataframe=True)
display(m_df.T)

print("--- df.attrs (agent-native metadata: legend + lineage + kind) ---")
pprint({k: (v[:200] + "…" if isinstance(v, str) and len(v) > 200 else v) for k, v in m_df.attrs.items()})

legend = m_df.attrs.get("legend") or m_df.attrs.get("riskmodels_semantic_cheatsheet")
if legend:
    print("--- full legend (truncated for display) ---")
    print(str(legend)[:1200] + ("…" if len(str(legend)) > 1200 else ""))


**Try it — `get_ticker_returns` (return time series)**

Semantic column names (e.g. `l3_market_hr`) match the docs. The cell below prints **`df.attrs`** again so you see metadata on a long panel, not only on snapshots.


In [ ]:
# ── 2a cont. — daily returns + rolling HR columns ───────────────────────────────────
from pprint import pprint

tr = client.get_ticker_returns(ticker_sdk, years=1)
display(tr.head(3))
print("columns sample:", list(tr.columns)[:14], "…")

print("--- df.attrs on ticker-returns DataFrame ---")
pprint({k: (v[:200] + "…" if isinstance(v, str) and len(v) > 200 else v) for k, v in tr.attrs.items()})


### 2b — AOM: subjects, lenses, and a **chain**

**Subject** = *what* (e.g. `stock("NVDA")`, or `portfolio_inline([...])` in the spec). **Lens** = *which analytic slice*: `return_attribution` (performance sleeves), `exposure` (snapshot of factor HR/ER), `risk_decomposition` (variance view), etc. The fluent builder compiles to HTTP steps; inspect `steps_out` after `run()`.

**First** cell: `return_attribution` + `timeseries` — “why did it move?” for **TSLA** YTD.

**Second** cell: a **chain** — `analyze(exposure snapshot)` then `hedge_action()`, which binds to `client.decompose` so you see *multi-step* reasoning (v1 executor is **stock**-first; batch portfolios stay in Part 3 REST / SDK).


In [ ]:
%matplotlib inline

from typing import Any

import os

import pandas as pd

os.environ.setdefault("RISKMODELS_HTTP_TIMEOUT", "300")

_req = (
    rm()
    .subject(stock("TSLA"))
    .scope(date_range_preset="ytd", as_of="latest")
    .return_attribution(resolution="full_stack", view="timeseries")
    .structured()  # required: turns the fluent builder into an AOMRequest dict
)
_out = run(client, _req)


def _first_ticker_returns_df(steps: list[dict[str, Any]]) -> pd.DataFrame | None:
    for step in steps:
        if step.get("op") != "rest_fetch" or step.get("client_method") != "get_ticker_returns":
            continue
        res = step.get("result")
        if isinstance(res, pd.DataFrame) and not res.empty:
            return res
    return None


_df = _first_ticker_returns_df(_out.get("steps_out") or [])
if _df is None or _df.empty:
    print("No ticker-returns DataFrame in steps_out:", _out.get("errors"))
else:
    _df = _df.sort_values("date").reset_index(drop=True)
    for cname in ("returns_gross", "l1_combined_factor_return", "l2_combined_factor_return", "l3_combined_factor_return"):
        if cname not in _df.columns:
            print(f"Missing {cname!r} — skip line chart.")
            break
    else:
        import matplotlib.pyplot as plt
        import numpy as np

        def _cum(x: pd.Series) -> np.ndarray:
            a = np.asarray(x.fillna(0.0), dtype=float)
            return np.cumprod(1.0 + a) - 1.0

        t = pd.to_datetime(_df["date"])
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(t, _cum(_df["returns_gross"]), label="TSLA", color="#60a5fa", linewidth=1.4)
        ax.plot(t, _cum(_df["l1_combined_factor_return"]), label="Market (L1 CFR)", color="#6366f1", linewidth=1.1)
        ax.plot(t, _cum(_df["l2_combined_factor_return"]), label="Sector (L2 CFR)", color="#34d399", linewidth=1.1)
        ax.plot(t, _cum(_df["l3_combined_factor_return"]), label="Subsector (L3 CFR)", color="#9ca3af", linewidth=1.1)
        ax.axhline(0, color="#444", linewidth=0.8, linestyle="--")
        ax.set_title("TSLA — cumulative sleeves (AOM)")
        ax.legend(frameon=False, fontsize=9)
        ax.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        plt.show()


In [ ]:
# ── 2b cont. — Chain: exposure snapshot → hedge_action (→ decompose) ─────────────
from pprint import pprint

from riskmodels.aom import analyze, hedge_action

_chain_ticker = "NVDA"

_chain_req = (
    rm()
    .subject(stock(_chain_ticker))
    .scope(as_of="latest")
    .chain(
        analyze(lens="exposure", resolution="full_stack", view="snapshot"),
        hedge_action(),
    )
    .structured()
)
_chain_out = run(client, _chain_req)

print("Compiled chain — op + client_method:")
for _step in _chain_out.get("steps_out") or []:
    print(" ", _step.get("op"), "|", _step.get("client_method") or _step.get("error") or "")

if _chain_out.get("errors"):
    pprint(_chain_out["errors"])

_hedge = None
for _step in _chain_out.get("steps_out") or []:
    if _step.get("op") == "hedge_action":
        _hedge = _step.get("result")
        break
if _hedge is not None:
    print("--- hedge_action (decompose) ---")
    if isinstance(_hedge, dict):
        pprint(_hedge)
    else:
        display(_hedge)


---
## Part 3 — REST cookbook (batch, rankings, PDF, L3 table)

You already used REST for **returns** in Part 1 and saw **metrics** via both REST and the SDK. This section is intentionally repetitive: one screen-friendly `session.get` per recipe so you can copy-paste into Bash, `curl`, or another stack.

Skim the headings — **Risk snapshot** = HR + ER + vol, **L3 decomposition** = risk through time, **portfolio hedge** = weighted HRs.


---
### Risk snapshot — hedge ratios (HR) + explained risk (ER)

`GET /metrics/{ticker}` — one row, many scalars. **Compare** to the SDK `get_metrics` output in Part 2: same numbers, but here you see the raw wire keys (`l3_mkt_hr`, …) nested under `metrics`.

Remember: **HR** ↔ how many dollars of each ETF for each dollar of stock. **ER** ↔ how much variance that factor sleeve explains (L3 residual closes the loop).


In [ ]:
# ── Risk Snapshot ────────────────────────────────────────────────────────────────────
ticker = "NVDA"  # Try it: change to "AAPL", "TSLA", "JPM"

resp = session.get(f"{BASE_URL}/metrics/{ticker}")
resp.raise_for_status()
m = resp.json()
metrics = m.get("metrics", m)
meta = m.get("meta", {})

snapshot = pd.DataFrame({"Value": {
    "Date":             m.get("teo"),
    "Price":            metrics.get("price_close"),
    "Vol (23d)":        round(metrics["vol_23d"], 4) if metrics.get("vol_23d") else None,
    "Sector ETF":       meta.get("sector_etf"),
    "Subsector ETF":    meta.get("subsector_etf"),
    "L1 Market HR":     round(metrics["l1_mkt_hr"], 4) if metrics.get("l1_mkt_hr") else None,
    "L1 Market ER":     round(metrics["l1_mkt_er"], 4) if metrics.get("l1_mkt_er") else None,
    "L2 Market HR":     round(metrics["l2_mkt_hr"], 4) if metrics.get("l2_mkt_hr") else None,
    "L2 Sector HR":     round(metrics["l2_sec_hr"], 4) if metrics.get("l2_sec_hr") else None,
    "L3 Market HR":     round(metrics["l3_mkt_hr"], 4) if metrics.get("l3_mkt_hr") else None,
    "L3 Sector HR":     round(metrics["l3_sec_hr"], 4) if metrics.get("l3_sec_hr") else None,
    "L3 Subsector HR":  round(metrics["l3_sub_hr"], 4) if metrics.get("l3_sub_hr") else None,
    "L3 Residual ER":   round(metrics["l3_res_er"], 4) if metrics.get("l3_res_er") else None,
}})
print(f"{ticker} — Risk Snapshot")
display(snapshot)

---
## Deep Dive Snapshot

A precomputed risk report as a PNG image — one API call, no rendering code.

In [ ]:
# ── Deep Dive Snapshot ──────────────────────────────────────────────────────────────
ticker = "NVDA"  # Try it: change to "AAPL", "MSFT", "GOOGL"

resp = session.get(f"{BASE_URL}/snapshot/{ticker}", params={"format": "png"})
if resp.status_code == 404:
    print(f"No snapshot for {ticker} (not yet in coverage).")
else:
    resp.raise_for_status()
    display(Image(data=resp.content))

---
## Portfolio Hedge

Submit weighted positions and get back hedge ratios at every level in one batch call.

In [ ]:
# ── Portfolio Hedge ──────────────────────────────────────────────────────────────────
# Try it: change tickers and weights to your own portfolio (weights should sum to 1.0)
portfolio = {
    "AAPL":  0.25,
    "MSFT":  0.20,
    "NVDA":  0.20,
    "GOOGL": 0.15,
    "AMZN":  0.10,
    "JPM":   0.10,
}

resp = session.post(
    f"{BASE_URL}/batch/analyze",
    json={"tickers": list(portfolio.keys()), "metrics": ["hedge_ratios"], "years": 1},
)
resp.raise_for_status()
results = resp.json()["results"]

rows = []
for ticker, weight in portfolio.items():
    r  = results.get(ticker, {})
    hr = r.get("hedge_ratios") or {}
    rows.append({
        "ticker":       ticker,
        "weight":       weight,
        "l1_market_hr": hr.get("l1_market"),
        "l2_market_hr": hr.get("l2_market"),
        "l2_sector_hr": hr.get("l2_sector"),
        "l3_market_hr": hr.get("l3_market"),
        "l3_sector_hr": hr.get("l3_sector"),
        "l3_sub_hr":    hr.get("l3_subsector"),
    })

df_port = pd.DataFrame(rows).set_index("ticker")

def _wtd(col):
    return round((df_port["weight"] * df_port[col].fillna(0)).sum(), 4)

port_summary = pd.DataFrame({
    "Market HR": [_wtd("l1_market_hr"), _wtd("l2_market_hr"), _wtd("l3_market_hr")],
    "Sector HR": [float("nan"), _wtd("l2_sector_hr"), _wtd("l3_sector_hr")],
    "Subsector HR": [float("nan"), float("nan"), _wtd("l3_sub_hr")],
}, index=["L1", "L2", "L3"]).rename_axis("Level")

print("Portfolio-level hedge ratios (weighted):")
display(port_summary)

print("\nPer-position breakdown:")
display(df_port[["weight", "l1_market_hr", "l2_market_hr",
                  "l2_sector_hr", "l3_market_hr", "l3_sector_hr", "l3_sub_hr"]])

---
## Rankings — Where Does Your Stock Rank?

Cross-sectional percentile rankings across the full universe, sector, or subsector.

In [ ]:
# ── Rankings ─────────────────────────────────────────────────────────────────────────
ticker = "NVDA"  # Try it: change to any ticker

resp = session.get(f"{BASE_URL}/rankings/{ticker}")
resp.raise_for_status()
data = resp.json()

if data.get("rankings"):
    df_rank = pd.DataFrame(data["rankings"])
    print(f"{data['ticker']} — Rankings (as of {data.get('date', 'latest')})\n")
    display(df_rank[["display_label", "rank_percentile", "rank_ordinal", "cohort_size"]].head(15))
else:
    print(f"No rankings for {ticker}")

---
## Factor Risk Decomposition

**Same endpoint** as the **L3 Plotly** stack near the start of Part 1 (`GET /l3-decomposition`); this cell shows the raw month grid as a table for readers who prefer numbers in rows.

Break down a stock's risk into market, sector, subsector, and residual components month by month.

In [ ]:
# ── Factor Risk Table ────────────────────────────────────────────────────────────────
ticker = "NVDA"  # Try it: change to any ticker

resp = session.get(
    f"{BASE_URL}/l3-decomposition",
    params={"ticker": ticker, "market_factor_etf": "SPY"},
)
resp.raise_for_status()
body = resp.json()

df_risk = pd.DataFrame({
    "date":         pd.to_datetime(body.get("dates", [])),
    "market_er":    body.get("l3_market_er", []),
    "sector_er":    body.get("l3_sector_er", []),
    "subsector_er": body.get("l3_subsector_er", []),
    "residual_er":  body.get("l3_residual_er", []),
})
df_risk[["market_er", "sector_er", "subsector_er", "residual_er"]] = (
    df_risk[["market_er", "sector_er", "subsector_er", "residual_er"]].fillna(0)
)
df_risk = df_risk.sort_values("date").reset_index(drop=True)

if df_risk.empty:
    print(f"No factor data for {ticker}.")
else:
    df_risk["total_er"] = df_risk[["market_er", "sector_er",
                                    "subsector_er", "residual_er"]].sum(axis=1)

    df_risk["month"] = df_risk["date"].dt.to_period("M")
    df_month = df_risk.groupby("month", as_index=False).last().drop(columns=["month"])

    pct_cols = ["market_er", "sector_er", "subsector_er", "residual_er", "total_er"]
    df_month[pct_cols] = (df_month[pct_cols] * 100).round(2)
    df_month.rename(columns={
        "market_er": "market_%",
        "sector_er": "sector_%",
        "subsector_er": "subsector_%",
        "residual_er": "residual_%",
        "total_er": "total_%",
    }, inplace=True)

    print(f"Factor risk attribution for {ticker} — most recent 12 months (month-end, L3 ER → %)")
    print(f"Market ETF: {body.get('market_factor_etf', '—')}  |  Universe: {body.get('universe', '—')}")
    print()
    display(df_month.tail(12))

---
## Next steps (and where to read more)

- **Contract / fields:** [riskmodels.app/docs/api](https://riskmodels.app/docs/api) and repo `SEMANTIC_ALIASES.md`.
- **Python SDK:** `pip install riskmodels-py`.
- **CLI & MCP (terminal installs, doctor, config):** [riskmodels.app/quickstart](https://riskmodels.app/quickstart) — keep analytical work in this notebook; use the CLI when you want `npx riskmodels …` tooling without Python.
- **AOM spec:** `aom/AOM_SPEC.md`.
- **Support:** [service@riskmodels.app](mailto:service@riskmodels.app)

You climbed wire → SDK → AOM; that is the same integration hardening path most teams follow.
